In [ ]:
'''CASMI26 | Validate And Predict
Run all cells in a fresh Kaggle session. Required inputs: competition data,
V4 fingerprint weights, COCONUT fingerprints, rank_train.npz and an offline
RDKit 2026.03.3 wheel when that version is not already installed.
Outputs: validation tables, submission.csv and test_diagnostics.csv.
Public inference components adapted from prvsiyan Analog Propagation V20.

Validation is a retrieval-exclusion diagnostic. The training membership of
public neural checkpoints and ranker rows is unverified: this is NOT clean OOF
and is not used to automatically select a submission model.
'''

In [ ]:
'''CASMI26 | Input Preflight
Resolve every required file before loading models. Duplicate files require an
explicit source choice instead of silently selecting an arbitrary version.
'''
from pathlib import Path
import json, time
INPUT_ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT.mkdir(parents=True, exist_ok=True)
def find(name):
    hits = sorted(INPUT_ROOT.rglob(name))
    if len(hits) != 1:
        raise RuntimeError(f'Expected one {name}; found {len(hits)}: {hits}. Check attached inputs.')
    return str(hits[0])

required = ['train.parquet', 'test.parquet', 'sample_submission.csv',
            'coco_fp.npy', 'coco_mass.npy', 'coco_meta.pkl', 'fp_bits.npy', 'rank_train.npz']
INPUTS = {name: find(name) for name in required}
FP_PATHS = sorted(str(p) for p in INPUT_ROOT.rglob('fp_*.pt')
                  if 'casmi26-fp-models-v4' in p.parts)
expected = {'fp_single_aug.pt', 'fp_single_s2.pt', 'fp_merged_m1.pt', 'fp_merged_m2.pt'}
assert len(FP_PATHS) == 4 and {Path(p).name for p in FP_PATHS} == expected, 'Attach complete V4 weights.'
manifest = {'files': INPUTS, 'weights': FP_PATHS, 'validation': 'training overlap unverified'}
(OUT / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''

# ===================================================================================
#  CONFIG — every knob in one place. Each is annotated with the measurement behind it.
# ===================================================================================
class CFG:
    # --- candidate generation -------------------------------------------------------
    PPM_WIN      = 10.0   # neutral-mass window for candidates. Tighter IS better -- up to the
                          # point where it starts deleting answers, which is the same trap as
                          # CAND_CAP below. Check window RECALL, not just MRR:
                          #   5.0 ppm 0.972 | 7.0 0.992 | 8.5 0.992 | 10.0 1.000 | 12+ 1.000
                          # 10 ppm is the smallest window that loses nothing, and everything
                          # wider only adds decoys. Full four-channel ranker agrees:
                          #   8.5 -> predLB 0.361 | 10.0 -> 0.371 | 12.0 -> 0.363
                          # (Much wider genuinely does hurt: +-20 -> 0.509, +-30 -> 0.500 C2 MRR.)
                          # timsTOF precursor error stays under ~9 ppm (+1.4 ppm offset).
    PPM_FALLBACK = 30.0   # only used if the tight window returns nothing at all.

    # --- spectrum cleaning ----------------------------------------------------------
    INT_FLOOR    = 0.002  # drop peaks below this fraction of the base peak
    MAX_PEAKS    = 256    # keep the N most intense peaks after the floor
    MZ_TOL       = 0.01   # Da tolerance when matching two peaks
    INT_POWER    = 1.0    # intensity transform before similarity (1.0 + entropy weighting
                          # beat sqrt: Class-1 0.919 vs 0.895)
    ENT_WEIGHT   = True   # Li et al. 2021 entropy weighting of low-entropy spectra

    # --- analog propagation (the main idea) -----------------------------------------
    ANALOG_WIN   = 200.0  # +- Da mass-shift window. +-400 gave no gain (0.520 vs 0.521).
    N_ANALOG     = 80     # analogs kept per molecule. Flat above 80 -- predicted LB 0.3683 (60),
                          # 0.3709 (80), 0.3705 (100), 0.3703 (140). Nothing to win here.
    SIM_POWER    = 3.0    # sim^p weighting. p=1 -> 0.498, p=3 -> 0.521, p=4 -> 0.525 on local
                          # validation -- but see the model-selection section: that validation set
                          # is the one the checkpoint was early-stopped on, so small local wins on
                          # it are not trustworthy. p=3 is the value that actually scored 0.335.

    # --- ranker ---------------------------------------------------------------------
    W1_PRIORS    = (0.30, 0.60)  # REVERTED. A narrow plateau (.40,.45,.50) scored 0.3789 in my
                          # own sweep and then LOST on the leaderboard (0.335 -> 0.330). The sweep
                          # was in-sample: it trained on all 819 query groups and evaluated on 250
                          # of them. See the ranker section -- hold out BY QUERY or you are just
                          # measuring capacity to memorise your own evaluation set.
    SEEDS        = (0, 1, 2, 3)  # seed alone moves the LB by ~0.006; bag several.
    W1           = 0.50   # weight on the Class-1 simulation. NOT the class share (0.16) --
                          # it is the leaderboard-calibrated value, chosen by 5-fold CV held out
                          # by query in make_ranker3.py.
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03,
               min_samples_leaf=80, l2_regularization=1.0)
               # Depth 6, also REVERTED from 10. In-sample, depth 10 looked worth +0.007; on the
               # leaderboard it was not. Deeper trees fit the evaluation queries better precisely
               # because those queries were in the training rows.
    USE_BIO_DB   = False  # add ChEBI + LIPID MAPS. Costs -0.026 Class-2 MRR in dilution but
                          # adds 7-19% coverage of in-library structures. Looked net-positive
                          # on validation but the leaderboard disagreed (0.299 -> 0.295), so it
                          # ships OFF. Flip it if your pool recall differs.
                          # (Adding all of PubChem instead costs -0.35: measured, do not.)
    CAND_CAP     = 500    # pure runtime guard on in-silico fragmentation (~14 ms/candidate).
                          # It is deliberately LARGE. A cap of 80 ranked by "library hit, then
                          # closest in mass" looks like an adaptive version of "tighter windows
                          # win" -- it is not, it is a recall bug. Class-2 answers have
                          # library_sim = 0 *by definition* (no reference spectrum exists), so
                          # they get ordered by mass alone, which inside a +-8.5 ppm window is
                          # arbitrary. Measured truth retention on the Class-2 holdout:
                          #   no cap 0.992 | cap 400 0.992 | cap 200 0.952 | cap 80 0.752
                          # i.e. a cap of 80 throws away a QUARTER of the reachable answers.
                          # Class 1 is untouched (0.992 at every cap) because lv*100 protects it,
                          # which is exactly why the bug survives casual validation.
                          # Median window is 52 candidates, max 401, so 500 essentially never
                          # fires -- and when it does it ranks by the model, not by mass.
    TOPN         = 25     # the metric allows 25 guesses; there is no penalty for using them all


In [ ]:
'''CASMI26 | Data Paths'''
import os, time, pickle, math
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa
T0 = time.time()
TRAIN, TEST, SAMPLE = (INPUTS[k] for k in ('train.parquet', 'test.parquet', 'sample_submission.csv'))


In [ ]:
'''CASMI26 | Chemistry Runtime
Install the metric-compatible RDKit before importing its native modules.
Use a fresh session for Run All.
'''
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
try:
    rdkit_version = version('rdkit')
except PackageNotFoundError:
    rdkit_version = ''
if rdkit_version not in ('2026.3.3', '2026.03.3'):
    if 'rdkit' in sys.modules:
        raise RuntimeError('Restart session before Run All: a different RDKit version is already loaded.')
    wheels = sorted(INPUT_ROOT.rglob('rdkit-2026.3.3-*.whl')) + sorted(INPUT_ROOT.rglob('rdkit-2026.03.3-*.whl'))
    if not wheels:
        raise RuntimeError('Attach an RDKit 2026.03.3 wheel compatible with this Python runtime.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', str(wheels[0])], check=True)
import rdkit
from rdkit import Chem
assert rdkit.__version__ in ('2026.3.3', '2026.03.3')
HAVE_RDKIT = True
print('RDKit:', rdkit.__version__)


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''
"""Similarity kernels: weighted cosine + spectral entropy similarity (Li et al. 2021)."""
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n=len(mz)
    if n==0: return np.empty(0,np.float32), np.empty(0,np.float32)
    mx=0.0
    for i in range(n):
        if it[i]>mx: mx=it[i]
    if mx<=0: return np.empty(0,np.float32), np.empty(0,np.float32)
    thr=floor*mx; c=0
    for i in range(n):
        if it[i]>=thr: c+=1
    idx=np.empty(c,np.int64); j=0
    for i in range(n):
        if it[i]>=thr: idx[j]=i; j+=1
    if c>topk:
        v=np.empty(c,np.float32)
        for i in range(c): v[i]=it[idx[i]]
        o=np.argsort(v)[c-topk:]
        k2=np.empty(topk,np.int64)
        for i in range(topk): k2[i]=idx[o[i]]
        k2.sort(); idx=k2; c=topk
    om=np.empty(c,np.float32); oi=np.empty(c,np.float32)
    s=0.0
    for i in range(c):
        om[i]=mz[idx[i]]; v=it[idx[i]]**power; oi[i]=v; s+=v
    if s>0:
        for i in range(c): oi[i]/=s
    if ent_weight:
        S=0.0
        for i in range(c):
            if oi[i]>0: S-=oi[i]*np.log(oi[i])
        if S<3.0:
            w=0.25+0.25*S; s2=0.0
            for i in range(c): oi[i]=oi[i]**w; s2+=oi[i]
            if s2>0:
                for i in range(c): oi[i]/=s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz)
    SA=0.0
    for x in range(n):
        if qp[x]>0: SA-=qp[x]*np.log(qp[x])
    SB=0.0
    for x in range(m):
        if cp[x]>0: SB-=cp[x]*np.log(cp[x])
    SAB=0.0; tot=0.0
    buf=np.empty(n+m,np.float64); b=0
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: buf[b]=qp[i]; i+=1; b+=1
        elif d>tol: buf[b]=cp[j]; j+=1; b+=1
        else: buf[b]=qp[i]+cp[j]; i+=1; j+=1; b+=1
    while i<n: buf[b]=qp[i]; i+=1; b+=1
    while j<m: buf[b]=cp[j]; j+=1; b+=1
    for x in range(b): tot+=buf[x]
    if tot<=0: return 0.0
    for x in range(b):
        v=buf[x]/tot
        if v>0: SAB-=v*np.log(v)
    return 1.0-(2.0*SAB-SA-SB)/np.log(4.0)

@njit(cache=True, fastmath=True)
def cos_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz); dot=0.0; na=0.0; nb=0.0
    for x in range(n): na+=qp[x]*qp[x]
    for x in range(m): nb+=cp[x]*cp[x]
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: i+=1
        elif d>tol: j+=1
        else: dot+=qp[i]*cp[j]; i+=1; j+=1
    if na<=0 or nb<=0: return 0.0
    return dot/np.sqrt(na*nb)

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]= entropy_sim(qmz,qp,cm,cp,tol) if kind==1 else cos_sim(qmz,qp,cm,cp,tol)
    return out

def prep(mz,it,floor=0.002,topk=256,power=0.5,ent_weight=False):
    return _clean(np.asarray(mz,np.float32),np.asarray(it,np.float32),floor,topk,power,ent_weight)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz,qp,cmz,cp,tol,shift):
    """Best of direct and mass-shifted entropy similarity."""
    a = entropy_sim(qmz,qp,cmz,cp,tol)
    if shift > -0.001 and shift < 0.001: return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)): sm[i]=cmz[i]+shift
    b = entropy_sim(qmz,qp,sm,cp,tol)
    return a if a>b else b

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind,shift):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]=entropy_sim_shift(qmz,qp,cm,cp,tol,shift[k])
    return out


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''

# ===================================================================================
#  Adduct -> neutral mass.  The instrument measures the *ion*; candidates are neutral
#  molecules, so every adduct has to be undone before we can compare masses.
# ===================================================================================

MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
            S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
            Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {
 "[M+H]+":(1,1,PROTON), "[M+NH4]+":(1,1,NH4-E), "[M+Na]+":(1,1,MASS['Na']-E),
 "[M+K]+":(1,1,MASS['K']-E), "[M-H2O+H]+":(1,1,PROTON-H2O), "[M-2H2O+H]+":(1,1,PROTON-2*H2O),
 "[M+2H]2+":(1,2,2*PROTON), "[M]+":(1,1,-E), "[M-H2O]+":(1,1,-E-H2O),
 "[M+CH3OH+H]+":(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 "[M+CH3CN+H]+":(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 "[M-H]-":(1,1,-PROTON), "[M-H2O-H]-":(1,1,-PROTON-H2O), "[M+CH2O2-H]-":(1,1,FORMATE-PROTON),
 "[M+C2H4O2-H]-":(1,1,ACETATE-PROTON), "[M+Cl]-":(1,1,MASS['Cl']+E), "[M]-":(1,1,E),
 "[M-2H]-":(1,2,-2*PROTON), "[M+Na-2H]-":(1,1,MASS['Na']-2*PROTON),
 "[2M+H]+":(2,1,PROTON), "[2M+Na]+":(2,1,MASS['Na']-E), "[2M+NH4]+":(2,1,NH4-E),
 "[2M+K]+":(2,1,MASS['K']-E), "[2M-H]-":(2,1,-PROTON), "[2M+CH2O2-H]-":(2,1,FORMATE-PROTON),
 "[2M+C2H4O2-H]-":(2,1,ACETATE-PROTON), "[2M+Na-2H]-":(2,1,MASS['Na']-2*PROTON),
 "[3M+H]+":(3,1,PROTON), "[3M-H]-":(3,1,-PROTON),
}
def neutral_mass(mz, adduct):
    out=np.full(len(mz), np.nan); ad=np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m=(ad==a)
        if m.any(): out[m]=(mz[m]*z-d)/n
    return out


# ===================================================================================
#  Library + candidate pool
# ===================================================================================
def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14','normalized_smiles','adduct','precursor_mz',
                                     'ms2_mzs','ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks(); itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    nm  = neutral_mass(prec, add); ok = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best: best[k] = s
    print(f'library: {len(off)-1:,} spectra / {len(best):,} structures  ({time.time()-t0:.0f}s)', flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target-tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target+tol, 'right')
    return L['order'][lo:hi]

def build_rep(L):
    """One representative spectrum per structure (the richest), sorted by neutral mass.
       Using 3 per structure was WORSE (0.49 vs 0.52): extra spectra raise the max similarity
       of irrelevant structures too, which flattens the discrimination."""
    npk = np.diff(L['off']); best = {}; ik = L['ik']
    for i in range(len(ik)):
        k = ik[i]
        if k and (k not in best or npk[i] > npk[best[k]]): best[k] = i
    rep = np.array(sorted(best.values()))
    nm = L['nm'][rep]; ok = np.isfinite(nm)
    rep = rep[ok]; nm = nm[ok]; key = ik[rep]
    o = np.argsort(nm)
    return rep[o], key[o], nm[o]

# ===================================================================================
#  The two evidence channels
# ===================================================================================
def clean(mz, it):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)

def lib_sim(L, specs, target):
    """CLASS 1: direct match against library spectra of the same neutral mass."""
    cand = lib_window(L, target, target*CFG.PPM_WIN/1e6)
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT, 1)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return agg

def analog_sim(L, specs, target, rep, rep_key, rep_nm):
    """CLASS 2: mass-SHIFTED match over a wide window. Relatives of the unknown fragment
       into the same ions offset by the mass difference, so they still match."""
    lo = np.searchsorted(rep_nm, target-CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target+CFG.ANALOG_WIN, 'right')
    cand = rep[lo:hi]
    if len(cand) == 0: return []
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey = rep_key[lo:hi]; agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, 1, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''

# ===================================================================================
#  Candidate pool = COCONUT (attached, CC-BY) U training structures (rebuilt here).
#  Only the COCONUT half is redistributable, so the other half is computed at runtime.
# ===================================================================================
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
from multiprocessing import Pool as MPool
RDLogger.DisableLog('rdApp.*')

BITS = np.load(find('fp_bits.npy'))
_g = {}
def _fp_init():
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([_g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class Pool:
    """Candidates + fingerprints, sorted by exact mass."""
    def __init__(s, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        s._fp = fp[o]; s.mass = mass[o]
        s.keys = np.asarray(keys, dtype=object)[o]
        s.smiles = np.asarray(smiles, dtype=object)[o]
        s.nbits = nbits
        s.k2i = {k: i for i, k in enumerate(s.keys)}
    def window(s, t, ppm):
        a = np.searchsorted(s.mass, t*(1-ppm/1e6), 'left')
        b = np.searchsorted(s.mass, t*(1+ppm/1e6), 'right')
        return np.arange(a, b)
    def fps(s, idx):
        return np.unpackbits(np.asarray(s._fp[idx]), axis=1)[:, :s.nbits]

def build_pool():
    t0 = time.time()
    d = os.path.dirname(find('coco_fp.npy'))
    cm = pickle.load(open(d + '/coco_meta.pkl', 'rb'))
    co_fp = np.load(d + '/coco_fp.npy'); co_mass = np.load(d + '/coco_mass.npy')
    co_keys = np.asarray(cm['keys'], dtype=object); co_smis = np.asarray(cm['smiles'], dtype=object)
    print(f'COCONUT: {len(co_mass):,} structures', flush=True)

    # ChEBI + LIPID MAPS: small, high-precision, and covers mammalian/lipid metabolites that a
    # plant/microbe-focused NP database misses. +8.8% candidates for +7-19% coverage of the
    # structures in the public spectral libraries (see the pool section).
    if CFG.USE_BIO_DB:
        try:
            bd = os.path.dirname(find('bio_fp.npy'))
            bm = pickle.load(open(bd + '/bio_meta.pkl', 'rb'))
            bi_fp = np.load(bd + '/bio_fp.npy'); bi_mass = np.load(bd + '/bio_mass.npy')
            co_fp = np.vstack([co_fp, bi_fp]); co_mass = np.concatenate([co_mass, bi_mass])
            co_keys = np.concatenate([co_keys, np.asarray(bm['keys'], dtype=object)])
            co_smis = np.concatenate([co_smis, np.asarray(bm['smiles'], dtype=object)])
            print(f'+ ChEBI/LIPID MAPS: {len(bi_mass):,} structures', flush=True)
        except FileNotFoundError:
            print('ChEBI/LIPID MAPS dataset not attached - skipping', flush=True)

    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    tr = tr[~tr.inchikey14.isin(set(co_keys))]
    print(f'training structures to fingerprint: {len(tr):,}  (~5 min)', flush=True)
    with MPool(4) as mp:
        res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
    ok = [i for i, r in enumerate(res) if r is not None]
    tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
    tr_mass = np.array([res[i][1] for i in ok])
    tr_keys = tr.inchikey14.values[ok]; tr_smi = tr.normalized_smiles.values[ok]

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_keys, tr_keys])
    smis = np.concatenate([co_smis, tr_smi])
    good = np.isfinite(mass)
    print(f'pool: {int(good.sum()):,} structures   ({time.time()-t0:.0f}s)', flush=True)
    return Pool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''
"""Single source of truth for candidate ranking features (used by local fitting AND the notebook).

Deliberately EXCLUDES any feature revealing pool provenance (src / np_likeness): in the Class-2
simulation the answer is always a training-library structure, so those columns leak.
"""
import numpy as np

N_ANALOG = 80
P_SIM    = 3.0
N_FEAT   = 31

def _rank_norm(x):
    o=np.argsort(-x); r=np.empty(len(x)); r[o]=np.arange(len(x)); return r/max(1,len(x)-1)

def _z(x):
    s=x.std()
    return (x-x.mean())/s if s>1e-9 else np.zeros_like(x)

def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):
    """cand_fp (nc,nbits), cand_lib (nc,) library similarity (0 if none),
       analog_fp (na,nbits), analog_sim (na,) descending,
       model_logits (nbits,) or None -> fingerprint-model evidence,
       frag (nc,) or None -> in-silico fragmentation explain-score (MetFrag-lite).
       Returns X (nc, N_FEAT)."""
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter/(cs[:,None]+asum[None,:]-inter+1e-9)
        w = np.clip(np.asarray(analog_sim,np.float32),0,None)
        ap = (tan*(w**P_SIM)[None,:]).max(1)
        a1 = (tan*w[None,:]).max(1)
        best_tan = tan.max(1); top_tan = tan[:,0]; top_sim = float(w[0])
        mean_tan = (tan*(w**P_SIM)[None,:]).sum(1)/((w**P_SIM).sum()+1e-9)
    else:
        ap=a1=best_tan=top_tan=mean_tan=np.zeros(nc,np.float32); top_sim=0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)       # exact Bayes LL up to a constant
        nrm = raw/np.sqrt(np.maximum(cs,1.0))                 # length-corrected variant
        mfeat = [_z(raw), _rank_norm(raw), raw-raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw==raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc,np.float32)]*6
    # cross-channel agreement: a genuine Class-1 hit should look good to the MODEL too.
    # When the library's best match also ranks high under f.z, the library evidence is
    # corroborated; when it does not, the library hit is probably a same-mass impostor.
    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0      # 1 = model also ranks it first
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv*(1.0-mr), ap*(1.0-mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree*lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0,1]) if lv.std()>1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc,np.float32)]*6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr-fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc,np.float32)]*4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc,lvmax), lv-lvmax, (lv>0).astype(float),
        ap, _rank_norm(ap), np.full(nc,apmax), ap-apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc,top_sim),
        np.full(nc, np.log(max(nc,1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''

# ===================================================================================
#  Channel 4: spectrum -> molecular fingerprint (CSI:FingerID-style), ranked by f . z
# ===================================================================================
import torch
_MODEL = None
def load_model():
    """Optional. If the weights dataset is not attached, everything still runs without it."""
    global _MODEL
    import glob
    paths = FP_PATHS
    if not paths:
        print('fingerprint model not attached - running with 3 channels'); return None
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    # Each model is fed the input distribution it was TRAINED on. 'fp_merged_*' saw fused
    # multi-spectrum inputs (--merge_p 0.6) and wants one merged peak list; 'fp_single_*' saw
    # one spectrum at a time. Feeding either the wrong way costs ~0.02 Class-2 MRR:
    #   m1: merged 0.4896 / per-spectrum 0.4687     s2: per-spectrum 0.4799 / merged 0.4597
    single, merged = [], []
    for pth in paths:
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        assert ck['nbits'] == len(BITS), 'Weight / fingerprint metadata mismatch'
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        (merged if 'merged' in pth.split('/')[-1] else single).append(net)
        print(f'  loaded {pth.split("/")[-1]}: d={ck["d"]} layers={ck["layers"]} step={ck.get("step")}')
    print(f'fingerprint models: {len(single)} single-input, {len(merged)} merged-input, on {dev}')
    _MODEL = (single, merged, dev, ck['nbits'])
    return _MODEL

def _merge_peaks(sub):
    """All of a molecule's peaks collapsed into one pseudo-spectrum (near-duplicate m/z merged,
       keeping the stronger peak). A second view of the same molecule."""
    mz = np.concatenate([np.asarray(r.ms2_mzs, float) for r in sub.itertuples()])
    it = np.concatenate([np.asarray(r.ms2_normalized_intensities, float) /
                         max(float(np.asarray(r.ms2_normalized_intensities, float).max()), 1e-9)
                         for r in sub.itertuples()])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j]-mz[j-1] < 0.005:
            if it[j] >= it[j-1]: keep[j-1] = False
            else: keep[j] = False
    return mz[keep], it[keep]

@torch.no_grad()
def model_logits(sub):
    """Two fusion views, averaged: (a) per-spectrum logits averaged, (b) one merged peak list.
       Measured on the Class-2 holdout: (a) 0.468, (b) 0.459, mean of both 0.475."""
    if _MODEL is None: return None
    single, merged, dev, nbits = _MODEL
    out = []
    if single:
        za = _logits_from(sub, single)
        if za is not None: out.append(za)
    if merged:
        mz, it = _merge_peaks(sub)
        r0 = next(sub.itertuples())
        zb = _logits_raw([(mz, it)], merged, float(np.median(sub.precursor_mz)), r0.adduct,
                         r0.instrument_type, 25.0,
                         float(np.mean([1.0 if m=='positive' else -1.0 for m in sub.ionization_mode])))
        if zb is not None: out.append(zb)
    return np.mean(out, axis=0) if out else None

@torch.no_grad()
def _logits_from(sub, nets):
    if _MODEL is None: return None
    dev = _MODEL[2]
    rows = list(sub.itertuples())
    P = [prep_peaks(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz)) for r in rows]
    valid = [(r, p) for r, p in zip(rows, P) if len(p[0])]
    rows = [r for r, p in valid]
    P = [p for r, p in valid]
    if not P: return None
    B = len(P); N = max(len(a) for a,_ in P)
    mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)
    for i,(a,b) in enumerate(P):
        mz[i,:len(a)]=a; it[i,:len(b)]=b; pad[i,:len(a)]=False
    def ce_of(r):
        v=r.collision_energy_ev
        try: return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
        except Exception: return 25.0
    T=lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad),
            T(np.array([float(r.precursor_mz) for r in rows[:B]],np.float32)),
            T(np.array([ADDUCT_IX.get(r.adduct, ADDUCT_IX['<unk>']) for r in rows[:B]])),
            T(np.array([instr_family(r.instrument_type) for r in rows[:B]])),
            T(np.array([ce_of(r) for r in rows[:B]],np.float32)),
            T(np.array([1.0 if r.ionization_mode=='positive' else -1.0 for r in rows[:B]],np.float32)))
    # average over the ensemble, then over the molecule's spectra
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

@torch.no_grad()
def _logits_raw(pairs, nets, prec, adduct, instrument, ce, mode):
    """Same forward pass for an explicitly supplied peak list."""
    if _MODEL is None: return None
    dev = _MODEL[2]
    P=[prep_peaks(mz, it, prec) for mz, it in pairs]
    P=[(a,b) for a,b in P if len(a)]
    if not P: return None
    B=len(P); N=max(len(a) for a,_ in P)
    mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)
    for i,(a,b) in enumerate(P):
        mz[i,:len(a)]=a; it[i,:len(b)]=b; pad[i,:len(a)]=False
    T=lambda x: torch.as_tensor(x, device=dev)
    args=(T(mz),T(it),T(pad), T(np.full(B,prec,np.float32)),
          T(np.full(B, ADDUCT_IX.get(adduct, ADDUCT_IX['<unk>']))),
          T(np.full(B, instr_family(instrument))),
          T(np.full(B, ce, np.float32)), T(np.full(B, mode, np.float32)))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)


In [ ]:
'''CASMI26 | Validation Lab

Reusable inference component for molecule-level validation.
'''

# ===================================================================================
#  Calibrated ranker.  Fitted here, in-notebook, from shipped simulation features so
#  the whole thing is reproducible and you can retune W1 in one line.
# ===================================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
z = np.load(find('rank_train.npz'))
NFEAT = z['X'].shape[1]
assert NFEAT == N_FEAT, f'Expected {N_FEAT} rank features, got {NFEAT}; attach matching rank_train.npz.'
assert np.isfinite(z['X']).all(), 'Non-finite ranker training features'

# ---------------------------------------------------------------------------------
#  ⚠️  HistGradientBoostingClassifier defaults to random_state=None.
#  I submitted the SAME notebook version twice and got 0.292 and 0.298 -- a 0.006
#  spread from the ranker's seed alone (5 local seeds span 0.2961-0.3033). If you are
#  chasing a 0.005 leaderboard difference in this competition, you may be chasing noise.
#  Fix: pin the seed AND average over several, which removes the variance and lifts the mean.
# ---------------------------------------------------------------------------------
RANKERS = []
for w1 in CFG.W1_PRIORS:                  # a NARROW plateau around the swept peak -- see CFG
    W = np.where(z['M'] == 0, w1, 1.0 - w1)
    for sd in CFG.SEEDS:
        m = HistGradientBoostingClassifier(random_state=sd, **CFG.GBM)
        m.fit(z['X'], z['Y'], sample_weight=W)
        RANKERS.append(m)

def rank_proba(X):
    """Averaged over seeds and class priors -> deterministic and lower variance."""
    return np.mean([m.predict_proba(X)[:, 1] for m in RANKERS], axis=0)

print(f'ranker: {len(RANKERS)} GBMs ({len(CFG.W1_PRIORS)} priors x {len(CFG.SEEDS)} seeds) '
      f'on {z["X"].shape[0]:,} rows x {NFEAT} features')


In [ ]:
'''CASMI26 | Validation Protocol
Build fixed molecule-level folds. Class 1 retains another spectrum of the
same structure in the library; Class 2 removes every spectrum of the answer
but keeps the answer as a COCONUT candidate.
'''
from collections import defaultdict
from pathlib import Path
import json
import pickle

VAL_SEED = 20260919
N_CLASS1 = 80
N_CLASS2 = 160
QUERY_SPECTRA_PER_CLASS1 = 2
REQUIRE_TIMSTOF = True
RUN_FINGERPRINT = True
RUN_ANALOG = True
RUN_CALIBRATED_RANKER = True
TOP_K = 25
BOOTSTRAPS = 2000
OUT = Path('/kaggle/working' if Path('/kaggle/working').exists() else '.')
OUT.mkdir(parents=True, exist_ok=True)

assert 1 <= QUERY_SPECTRA_PER_CLASS1
assert N_CLASS1 > 0 and N_CLASS2 > 0
print(json.dumps({
    'seed': VAL_SEED, 'class1_keys': N_CLASS1, 'class2_keys': N_CLASS2,
    'query_spectra_per_class1': QUERY_SPECTRA_PER_CLASS1,
    'require_timsTOF': REQUIRE_TIMSTOF, 'fingerprint': RUN_FINGERPRINT,
    'analog': RUN_ANALOG, 'ranker': RUN_CALIBRATED_RANKER,
}, indent=2))
VALIDATION_SECONDS = 3600
print('Validation limit: one hour between molecules; complete test prediction follows.')


In [ ]:
'''CASMI26 | Load Reference Library
The library is loaded once. All later validation cells reuse this exact object,
so a fold changes evidence availability rather than the underlying data.
'''
L = load_library(TRAIN)
assert len(L['ik']) > 0
print('reference library loaded:', len(L['ik']), 'spectra')

In [ ]:
'''CASMI26 | Fixed Folds
The split key is InChIKey14, never an individual spectrum. A cached CSV makes
every later ablation evaluate exactly the same molecules.
'''
META_COLUMNS = [
    'inchikey14', 'normalized_smiles', 'instrument_type', 'adduct',
    'precursor_mz', 'ionization_mode', 'collision_energy_ev', 'num_peaks',
]
meta = pq.read_table(TRAIN, columns=META_COLUMNS).to_pandas()
assert len(meta) > 0 and meta.inchikey14.notna().all()
assert len(meta) == len(L['ik']), 'train row order changed between metadata and library loading'

instrument = meta.instrument_type.fillna('').astype(str).str.lower()
is_tims = instrument.str.contains('timstof')
eligible_rows = is_tims if REQUIRE_TIMSTOF else np.ones(len(meta), dtype=bool)
if eligible_rows.sum() == 0:
    raise RuntimeError('No timsTOF training spectra found; set REQUIRE_TIMSTOF = False only for a diagnostic run.')

coco_dir = os.path.dirname(find('coco_meta.pkl'))
coco_meta = pickle.load(open(os.path.join(coco_dir, 'coco_meta.pkl'), 'rb'))
coco_keys = set(np.asarray(coco_meta['keys'], dtype=object))
meta['_eligible'] = eligible_rows
meta['_row'] = np.arange(len(meta), dtype=np.int64)

grouped = meta[meta._eligible].groupby('inchikey14', sort=False)
rows_by_key = {
    key: block.sort_values(['num_peaks', '_row'], ascending=[False, True])._row.to_numpy(dtype=np.int64)
    for key, block in grouped
    if key and len(block) >= 1
}
all_keys = np.array(sorted(rows_by_key), dtype=object)
class1_pool = np.array([k for k in all_keys if len(rows_by_key[k]) >= 2], dtype=object)
rng = np.random.default_rng(VAL_SEED)

class1_keys = rng.permutation(class1_pool)[:min(N_CLASS1, len(class1_pool))]
class2_pool = np.array(sorted(set(all_keys).intersection(coco_keys) - set(class1_keys)), dtype=object)
class2_keys = rng.permutation(class2_pool)[:min(N_CLASS2, len(class2_pool))]

if len(class1_keys) < N_CLASS1 or len(class2_keys) < N_CLASS2:
    print(f'warning: requested C1={N_CLASS1}, C2={N_CLASS2}; available C1={len(class1_keys)}, C2={len(class2_keys)}')

fold_rows = []
for key in class1_keys:
    ids = rows_by_key[key]
    query_ids = ids[:min(QUERY_SPECTRA_PER_CLASS1, len(ids) - 1)]
    for row_id in query_ids:
        fold_rows.append(('class1', key, int(row_id)))
for key in class2_keys:
    for row_id in rows_by_key[key]:
        fold_rows.append(('class2', key, int(row_id)))

folds = pd.DataFrame(fold_rows, columns=['scenario', 'inchikey14', 'row_id'])
assert folds.row_id.is_unique
assert not set(class1_keys).intersection(class2_keys)
assert set(class2_keys).issubset(coco_keys), 'Class 2 answer must be a COCONUT candidate.'
folds.to_csv(OUT / 'validation_folds.csv', index=False)
print(f'folds: {len(class1_keys)} Class 1 structures / {len(class2_keys)} Class 2 structures / {len(folds)} query spectra')
display(folds.groupby('scenario').agg(structures=('inchikey14', 'nunique'), spectra=('row_id', 'size')))

In [ ]:
'''CASMI26 | Query Molecules And Retrieval Exclusions
A query molecule can contain multiple spectra. Class 1 removes only those
query spectra from direct search. Class 2 removes its entire structure from
every reference channel.
'''
def spectrum_at(row_id):
    a, b = int(L['off'][row_id]), int(L['off'][row_id + 1])
    return L['mz'][a:b], L['it'][a:b]

query_groups = []
for (scenario, key), block in folds.groupby(['scenario', 'inchikey14'], sort=True):
    ids = block.row_id.to_numpy(dtype=np.int64)
    sub = meta.iloc[ids].copy()
    sub['ms2_mzs'] = [spectrum_at(i)[0] for i in ids]
    sub['ms2_normalized_intensities'] = [spectrum_at(i)[1] for i in ids]
    neutral = neutral_mass(sub.precursor_mz.to_numpy(float), sub.adduct.to_numpy(object))
    neutral = neutral[np.isfinite(neutral)]
    if len(neutral):
        query_groups.append(dict(
            scenario=scenario, key=key, row_ids=ids, sub=sub,
            target_mass=float(np.median(neutral)),
        ))

assert query_groups, 'No valid query molecules after neutral-mass conversion.'
class2_key_set = set(class2_keys)
eval_key_set = set(class1_keys).union(class2_keys)

def reps_without(excluded_keys):
    keep = ~np.isin(L['ik'], list(excluded_keys))
    n_peaks = np.diff(L['off'])
    best = {}
    for row_id in np.flatnonzero(keep):
        key = L['ik'][row_id]
        if key and (key not in best or n_peaks[row_id] > n_peaks[best[key]]):
            best[key] = row_id
    reps = np.array(list(best.values()), dtype=np.int64)
    masses = L['nm'][reps]
    ok = np.isfinite(masses)
    reps, masses = reps[ok], masses[ok]
    order = np.argsort(masses)
    return reps[order], L['ik'][reps][order], masses[order]

# Remove every evaluated structure from analog search: no hidden same-structure shortcut.
REP, REP_KEY, REP_NM = reps_without(eval_key_set)
print(f'analog references: {len(REP):,} representative spectra after removing {len(eval_key_set):,} query structures')

def direct_scores(specs, target_mass, excluded_rows):
    ids = lib_window(L, target_mass, target_mass * CFG.PPM_WIN / 1e6)
    if len(ids) == 0:
        return {}
    ids = ids[~np.isin(ids, list(excluded_rows))]
    scores = {}
    for mz, intensity in specs:
        qm, qi = clean(mz, intensity)
        if not len(qm):
            continue
        values = search(qm, qi, ids, L['off'], L['mz'], L['it'],
                        CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                        CFG.INT_POWER, CFG.ENT_WEIGHT, 1)
        for row_id, value in zip(ids, values):
            key = L['ik'][row_id]
            scores[key] = max(scores.get(key, -np.inf), float(value))
    return scores

def analog_scores(specs, target_mass):
    lo = np.searchsorted(REP_NM, target_mass - CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(REP_NM, target_mass + CFG.ANALOG_WIN, 'right')
    ids, keys = REP[lo:hi], REP_KEY[lo:hi]
    if not len(ids):
        return {}
    shifts = (target_mass - REP_NM[lo:hi]).astype(np.float32)
    scores = {}
    for mz, intensity in specs:
        qm, qi = clean(mz, intensity)
        if not len(qm):
            continue
        values = search_shift(qm, qi, ids, L['off'], L['mz'], L['it'],
                              CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                              CFG.INT_POWER, CFG.ENT_WEIGHT, 1, shifts)
        for key, value in zip(keys, values):
            scores[key] = max(scores.get(key, -np.inf), float(value))
    return dict(sorted(scores.items(), key=lambda item: -item[1])[:CFG.N_ANALOG])

In [ ]:
'''CASMI26 | Evaluate All Channels
Every candidate list is mass-valid first. The only difference between methods
is the ordering, so MRR changes are interpretable.
'''
pool = build_pool()
assert pool.nbits == BITS.size, 'Candidate fingerprints and V4 bit selection disagree.'
if RUN_FINGERPRINT:
    load_model()
if RUN_CALIBRATED_RANKER:
    assert RANKERS, 'Calibrated ranker did not fit.'

def rank_of_truth(keys, truth):
    where = np.flatnonzero(np.asarray(keys, dtype=object) == truth)
    return int(where[0] + 1) if len(where) else TOP_K + 1

def ordered_keys(candidate_ids, values, mass_error):
    order = np.lexsort((mass_error, -np.asarray(values, dtype=float)))
    return pool.keys[candidate_ids[order]], order

def predict_group(group):
    candidate_ids = pool.window(group['target_mass'], CFG.PPM_WIN)
    if not len(candidate_ids):
        candidate_ids = pool.window(group['target_mass'], CFG.PPM_FALLBACK)
    if not len(candidate_ids):
        return candidate_ids, {}, np.empty(0), {}, {}
    specs = list(zip(group['sub'].ms2_mzs, group['sub'].ms2_normalized_intensities))
    excluded_rows = set(group.get('row_ids', []))
    if group['scenario'] == 'class2':
        excluded_rows = set(np.flatnonzero(L['ik'] == group['key']))
    library = direct_scores(specs, group['target_mass'], excluded_rows)
    analog = analog_scores(specs, group['target_mass']) if RUN_ANALOG else {}
    candidate_fp = pool.fps(candidate_ids)
    candidate_keys = pool.keys[candidate_ids]
    mass_error = np.abs(pool.mass[candidate_ids] - group['target_mass'])

    library_score = np.array([library.get(key, 0.0) for key in candidate_keys], np.float32)
    analog_keys = [key for key in analog if key in pool.k2i]
    analog_fp = pool.fps(np.array([pool.k2i[key] for key in analog_keys], dtype=np.int64)) if analog_keys else None
    analog_score = np.array([analog[key] for key in analog_keys], np.float32)

    logits = model_logits(group['sub']) if RUN_FINGERPRINT else None
    fingerprint_score = candidate_fp @ logits if logits is not None else np.zeros(len(candidate_ids), np.float32)
    X = rank_features(candidate_fp, library_score, analog_fp, analog_score, logits)
    assert X.shape[1] == NFEAT, f'feature contract mismatch: {X.shape[1]} vs {NFEAT}'
    assert np.isfinite(X).all(), 'Non-finite candidate features'
    ranker_score = rank_proba(X) if RUN_CALIBRATED_RANKER else np.zeros(len(candidate_ids), np.float32)

    methods = {
        'mass': -mass_error,
        'library': library_score,
        'analog': X[:, 5],
        'fingerprint': fingerprint_score,
        'ranker': ranker_score,
    }
    return candidate_ids, methods, mass_error, library, analog

def evaluate_group(group):
    candidate_ids, methods, mass_error, library, analog = predict_group(group)
    tight = pool.window(group['target_mass'], CFG.PPM_WIN)
    rows = []
    for method in ('mass', 'library', 'analog', 'fingerprint', 'ranker'):
        keys = []
        if len(candidate_ids):
            keys, _ = ordered_keys(candidate_ids, methods[method], mass_error)
            keys = list(dict.fromkeys(keys))
        rank = rank_of_truth(keys[:TOP_K], group['key'])
        rows.append(dict(
            scenario=group['scenario'], inchikey14=group['key'],
            n_spectra=len(group['row_ids']), target_mass=group['target_mass'],
            n_candidates=len(candidate_ids), method=method, rank=rank,
            reciprocal_rank=0.0 if rank > TOP_K else 1.0 / rank,
            answer_in_10ppm=int(group['key'] in set(pool.keys[tight])),
            best_library=max(library.values(), default=0.0),
            best_analog=max(analog.values(), default=0.0),
            best_fingerprint=float(np.max(methods['fingerprint'])) if methods else 0.0,
        ))
    return rows

records = []
validation_started = time.monotonic()
rng = np.random.default_rng(VAL_SEED)
query_groups = [query_groups[i] for i in rng.permutation(len(query_groups))]
for index, group in enumerate(query_groups, 1):
    if records and time.monotonic() - validation_started > VALIDATION_SECONDS:
        print('Validation budget reached; remaining molecules are not evaluated.')
        break
    records.extend(evaluate_group(group))
    pd.DataFrame(records).to_csv(OUT / 'validation_results.csv', index=False)
    print(f'validated {index}/{len(query_groups)} molecules', flush=True)

results = pd.DataFrame(records)
assert results.groupby(['scenario', 'inchikey14', 'method']).size().eq(1).all()
manifest['validation_completed'] = len(records) // 5
manifest['validation_planned'] = len(query_groups)
manifest['selection_policy'] = 'fixed public ranker; diagnostic scores do not select the model'
(OUT / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
'''CASMI26 | Scorecard And Confidence
Intervals describe sampled query uncertainty only, not training overlap or
hidden-test shift. They do not establish an unbiased OOF score.
'''
def bootstrap_mean(values, seed):
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(values), size=(BOOTSTRAPS, len(values)))
    samples = values[draws].mean(axis=1)
    return float(values.mean()), float(np.quantile(samples, .025)), float(np.quantile(samples, .975))

summary = []
for (scenario, method), frame in results.groupby(['scenario', 'method'], sort=True):
    mean, lo, hi = bootstrap_mean(frame.reciprocal_rank.to_numpy(), VAL_SEED)
    summary.append(dict(
        scenario=scenario, method=method, molecules=len(frame),
        candidate_recall=frame.answer_in_10ppm.mean(),
        top1=(frame['rank'] == 1).mean(),
        recall_at_5=(frame['rank'] <= 5).mean(),
        recall_at_25=(frame['rank'] <= TOP_K).mean(),
        mrr_at_25=mean, mrr_ci_low=lo, mrr_ci_high=hi,
    ))
summary = pd.DataFrame(summary).sort_values(['scenario', 'mrr_at_25'], ascending=[True, False])
summary.to_csv(OUT / 'validation_summary.csv', index=False)
display(summary.style.format({
    'candidate_recall': '{:.1%}', 'top1': '{:.1%}',
    'recall_at_5': '{:.1%}', 'recall_at_25': '{:.1%}',
    'mrr_at_25': '{:.4f}', 'mrr_ci_low': '{:.4f}', 'mrr_ci_high': '{:.4f}',
}))

wide = results.pivot(index=['scenario', 'inchikey14'], columns='method', values='reciprocal_rank')
for scenario, frame in wide.groupby(level='scenario'):
    if {'ranker', 'analog'}.issubset(frame.columns):
        delta = (frame['ranker'] - frame['analog']).dropna().to_numpy()
        mean, lo, hi = bootstrap_mean(delta, VAL_SEED + 1)
        print(f'{scenario}: ranker - analog = {mean:+.4f} [{lo:+.4f}, {hi:+.4f}]')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for scenario, frame in results.groupby('scenario'):
    order = frame.groupby('method').reciprocal_rank.mean().sort_values().index
    axes[0].barh([f'{scenario}: {m}' for m in order],
                 frame.groupby('method').reciprocal_rank.mean().reindex(order),
                 label=scenario, alpha=.8)
axes[0].set_xlabel('MRR@25'); axes[0].set_title('Method score by validation scenario')
for method, frame in results.groupby('method'):
    axes[1].scatter(frame.n_candidates, frame.reciprocal_rank, s=10, alpha=.35, label=method)
axes[1].set_xscale('log'); axes[1].set_xlabel('mass-valid candidates'); axes[1].set_ylabel('reciprocal rank')
axes[1].set_title('Where ranking becomes difficult')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
for axis in axes:
    axis.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
'''CASMI26 | Error Gallery And Checks
The gallery reports failures as candidate absence versus bad ordering. This is
the decision point for candidate generation versus a better ranker.
'''
ranker_rows = results[results.method == 'ranker'].copy()
worst = ranker_rows.sort_values(['rank', 'n_candidates'], ascending=[False, False]).head(30)
best = ranker_rows.sort_values(['reciprocal_rank', 'n_candidates'], ascending=[False, True]).head(30)
display(worst[['scenario', 'inchikey14', 'rank', 'n_candidates', 'n_spectra',
               'best_library', 'best_analog', 'best_fingerprint']])
display(best[['scenario', 'inchikey14', 'rank', 'n_candidates', 'n_spectra',
              'best_library', 'best_analog', 'best_fingerprint']])

print(f'mass-window candidate recall: {results.answer_in_10ppm.mean():.1%}')
assert results['rank'].between(1, TOP_K + 1).all()
assert (OUT / 'validation_folds.csv').exists()
assert (OUT / 'validation_results.csv').exists()
assert (OUT / 'validation_summary.csv').exists()
(OUT / 'validation_config.json').write_text(json.dumps({
    'seed': VAL_SEED, 'n_class1': N_CLASS1, 'n_class2': N_CLASS2,
    'ppm': CFG.PPM_WIN, 'top_k': TOP_K, 'bootstraps': BOOTSTRAPS,
    'channels': ['mass', 'library', 'analog', 'fingerprint', 'ranker'],
}, indent=2) + '\n')
print('Validation checks passed. Outputs:', *sorted(path.name for path in OUT.glob('validation_*')), sep='\n  ')

In [ ]:
'''CASMI26 | Predict Current Test
Restore the full reference library for inference. Reuse the fitted rankers,
loaded neural models and candidate pool. No validation exclusions reach test.
'''
from functools import lru_cache
from rdkit.Chem.MolStandardize import rdMolStandardize
_tautomer = rdMolStandardize.TautomerEnumerator()

@lru_cache(maxsize=100000)
def metric_key(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    key = Chem.MolToInchiKey(_tautomer.Canonicalize(mol))
    return key.split('-')[0] if key else None

def unique_smiles(smiles, limit=25):
    result, seen = [], set()
    for smi in smiles:
        key = metric_key(str(smi))
        if key and key not in seen:
            result.append(str(smi))
            seen.add(key)
            if len(result) == limit:
                break
    return result

def validate_submission(frame, expected_ids):
    assert list(frame.columns) == ['molecule_id', 'smiles']
    assert frame.molecule_id.is_unique
    assert frame.molecule_id.tolist() == list(expected_ids)
    assert frame.smiles.notna().all()
    for value in frame.smiles:
        parts = value.split(';')
        assert 1 <= len(parts) <= TOP_K and all(parts)
        keys = [metric_key(s) for s in parts]
        assert all(keys) and len(set(keys)) == len(keys)

REP, REP_KEY, REP_NM = build_rep(L)
test = pq.read_table(TEST).to_pandas()
assert test.molecule_id.notna().all()
test['molecule_id'] = test.molecule_id.astype(str)
test_ids = test.molecule_id.drop_duplicates().tolist()
sample = pd.read_csv(SAMPLE, dtype={'molecule_id': str})
# Hidden reruns may not replace the downloadable sample. Test IDs are authoritative.
output_ids = sample.molecule_id.tolist() if (
    sample.molecule_id.is_unique and set(sample.molecule_id) == set(test_ids)
) else test_ids

predictions, test_diag = {}, []
for index, (mid, sub) in enumerate(test.groupby('molecule_id', sort=False), 1):
    masses = neutral_mass(sub.precursor_mz.to_numpy(float), sub.adduct.to_numpy(object))
    masses = masses[np.isfinite(masses) & (masses > 0)]
    if not len(masses):
        raise ValueError(f'No supported neutral mass for {mid}: {sub.adduct.unique()}')
    target = float(np.median(masses))
    group = dict(scenario='test', sub=sub, target_mass=target, row_ids=[])
    ids, scores, errors, library, analog = predict_group(group)
    fallback = not len(ids)
    if fallback:
        # ponytail: nearest mass is a last-resort guess, not spectral evidence.
        pos = int(np.searchsorted(pool.mass, target))
        ids = np.arange(max(0, pos - 100), min(len(pool.mass), pos + 100))
        order = np.argsort(np.abs(pool.mass[ids] - target), kind='stable')
    else:
        _, order = ordered_keys(ids, scores['ranker'], errors)
    guesses = unique_smiles(pool.smiles[ids[order]], TOP_K)
    if not guesses:
        raise ValueError(f'No chemically valid candidates for {mid}')
    predictions[mid] = ';'.join(guesses)
    test_diag.append(dict(molecule_id=mid, n_candidates=len(ids),
                          n_guesses=len(guesses), nearest_mass_fallback=fallback,
                          best_library=max(library.values(), default=0.0)))
    if index % 10 == 0:
        print(f'test: {index}/{len(test_ids)}', flush=True)

submission = pd.DataFrame({
    'molecule_id': output_ids,
    'smiles': [predictions[mid] for mid in output_ids],
})
validate_submission(submission, output_ids)
destination = OUT / 'submission.csv'
temporary = OUT / 'submission.csv.tmp'
submission.to_csv(temporary, index=False)
validate_submission(pd.read_csv(temporary, dtype=str, keep_default_na=False), output_ids)
temporary.replace(destination)
pd.DataFrame(test_diag).to_csv(OUT / 'test_diagnostics.csv', index=False)
print('Ready:', destination, submission.shape)
display(submission.head())


In [ ]:
'''CASMI26 | Runnable Output Checks'''
assert len(unique_smiles(['CC(=O)O', 'OC(C)=O', 'invalid_smiles'])) == 1
assert rank_of_truth(['A', 'B'], 'B') == 2
assert rank_of_truth(['A'], 'B') == TOP_K + 1
validate_submission(submission, output_ids)
print('Completed validation and current-test prediction. Submit submission.csv if desired.')
